# FT-900 训练期原始月报采集

只读来源阶段。144 个官方月份 URL 的冻结映射，4 份已归档原件复用，最多 140 次新 GET。每次请求前记录 started，未知完成状态禁止自动重试；完成的每年源字节打包 NAS 并远端哈希核对。无 SOXX 结果、因子、拟合、账户。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, tarfile, tempfile, urllib.request, shlex
from src.data.nas_market_data import _settings, nas_api_call, _ssh
root=Path.cwd();card=root/'research/census_ft900_semiconductor_export_source_20261023'
plan=json.loads((card/'full_training_source_plan.json').read_text());manifest_path=card/plan['url_manifest'];manifest=json.loads(manifest_path.read_text())
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest()==plan['url_manifest_sha256']
assert len(manifest['month_urls'])==plan['budget']['target_reference_months']==144
assert len(manifest['reuse'])==plan['budget']['reused_exact_PDFs']==4
store=root/'data/snapshots/census_ft900_training_20261023';store.mkdir(parents=True,exist_ok=True)
ledger_path=card/'full_training_request_ledger.jsonl';archive_path=card/'full_training_year_archives.jsonl'
def append(path,record):
    with path.open('a') as f:
        f.write(json.dumps(record,sort_keys=True)+'\n');f.flush();os.fsync(f.fileno())
def read_jsonl(path):
    return [json.loads(x) for x in path.read_text().splitlines()] if path.exists() else []
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
old=read_jsonl(ledger_path);finished={x['month']:x for x in old if x['event'] in ('completed','reused')}
started={x['month'] for x in old if x['event']=='started'}
assert len(finished)==len(set(finished)) and all(m in manifest['month_urls'] for m in finished)
if started-set(finished):raise RuntimeError('uncertain prior HTTP request, stop without retry: '+str(sorted(started-set(finished))))
settings=_settings(root)
archives={x['year']:x for x in read_jsonl(archive_path) if x.get('event')=='archive_verified'}
new_requests=sum(1 for x in old if x['event']=='started')
for year in range(2011,2023):
    months=[f'{year:04d}-{m:02d}' for m in range(1,13)]
    if str(year) in archives:
        assert all(m in finished for m in months)
        print(year,'already archived',flush=True);continue
    failure=None
    for month in months:
        url=manifest['month_urls'][month];name=url.rsplit('/',1)[-1];dest=store/name
        if month in finished:
            assert dest.exists() and sha(dest)==finished[month]['sha256']
            continue
        if month in manifest['reuse']:
            source=root/manifest['reuse'][month]['local_path'];expected=manifest['reuse'][month]['sha256']
            assert sha(source)==expected
            if not dest.exists():shutil.copyfile(source,dest)
            assert sha(dest)==expected
            record={'event':'reused','month':month,'url':url,'local_path':str(dest.relative_to(root)),'sha256':expected,'bytes':dest.stat().st_size,'source_path':str(source.relative_to(root))}
            append(ledger_path,record);finished[month]=record;continue
        if new_requests>=plan['budget']['new_monthly_PDF_GETs_max']:raise RuntimeError('frozen GET budget exhausted')
        start={'event':'started','month':month,'url':url,'requested_at_utc':datetime.now(timezone.utc).isoformat()}
        append(ledger_path,start);new_requests+=1
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'SOXL-Qlib-source-audit/1.0','Accept':'application/pdf'})
            with urllib.request.urlopen(req,timeout=60) as response:
                status=response.status;headers={k:response.headers.get(k) for k in ('Date','Last-Modified','ETag','Content-Type','Content-Length','Cache-Control')}
                payload=response.read(4000001)
            if status!=200 or len(payload)>4000000 or not payload.startswith(b'%PDF-'):raise ValueError('invalid_status_size_or_PDF_header')
            temporary=dest.with_suffix('.pdf.partial');temporary.write_bytes(payload);os.replace(temporary,dest)
            record={'event':'completed','month':month,'url':url,'requested_at_utc':start['requested_at_utc'],'received_at_utc':datetime.now(timezone.utc).isoformat(),'http_status':status,'headers':headers,'local_path':str(dest.relative_to(root)),'sha256':hashlib.sha256(payload).hexdigest(),'bytes':len(payload)}
            append(ledger_path,record);finished[month]=record
        except Exception as exc:
            failure={'event':'failed','month':month,'url':url,'error_type':type(exc).__name__,'error':str(exc),'observed_at_utc':datetime.now(timezone.utc).isoformat()}
            append(ledger_path,failure);break
    present=[m for m in months if m in finished]
    if present:
        with tempfile.TemporaryDirectory(prefix='ft900-year-') as temp:
            archive=Path(temp)/f'{year}.tar.gz'
            with tarfile.open(archive,'w:gz') as out:
                for month in present:
                    item=finished[month];src=root/item['local_path'];assert sha(src)==item['sha256']
                    out.add(src,arcname=src.name)
            digest=sha(archive);size=archive.stat().st_size
            receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
            assert receipt['sha256']==digest and receipt['size_bytes']==size
            remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
            assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
            row={'event':'archive_verified','year':str(year),'months':present,'archive_sha256':digest,'archive_bytes':size,'nas_artifact':receipt,'remote_hash_verified':True}
            append(archive_path,row);archives[str(year)]=row
            print(year,len(present),'PDFs, NAS verified',flush=True)
    if failure:raise RuntimeError('source GET failed and no retry authorized: '+json.dumps(failure))
assert len(finished)==144 and new_requests==140 and len(archives)==12
result={'status':'all_144_training_month_PDF_bytes_archived_source_only','target_months':144,'reused_PDFs':4,'new_HTTP_GETs':new_requests,'yearly_NAS_archives':len(archives),'all_years_remote_hash_verified':True,'historical_first_public_bytes_qualified':False,'Exhibit7_row_coverage_qualified':False,'classification_continuity_qualified':False,'market_outcome_reads':0,'factor_values':0,'model_fits':0,'native_accounts':0,'SOXX_goal_achieved':False}
(card/'full_training_source_result.json').write_text(json.dumps(result,indent=2)+'\n')
print(result['status'],flush=True)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2011 12 PDFs, NAS verified


2012 12 PDFs, NAS verified


2013 12 PDFs, NAS verified


2014 12 PDFs, NAS verified


...

2019 12 PDFs, NAS verified


2020 12 PDFs, NAS verified


2021 12 PDFs, NAS verified


2022 12 PDFs, NAS verified


all_144_training_month_PDF_bytes_archived_source_only


## 结论边界

本 notebook 的成功仅证明当前取得的训练期月报 PDF 原始字节及 NAS 备份。历史首次公开版本、Exhibit 7 全月行存在、2012 分类连续性、Stage1 增量信息与 0bp SOXX 账户均须分别核查。

本次执行完成 144/144 份训练期 PDF 字节（4 份复用，140 次新 GET），12 个年度 NAS 源包远端哈希通过；每年一个 PDF 从 NAS 限定恢复、12/12 哈希与尺寸匹配。144 份本地源对象按收据复核。没有市场结果、因子、拟合或账户。